# START: Liz Choi

## Hyperparameter Tuning for XGBoost Regressor

This section retunes the XGBoost regressor after incorporating Marko's feedback.

The main correction is that percentage-based metrics such as MAPE and MdAPE must be computed on the **actual ClosePrice scale**, not on the **LogClosePrice scale**.  
The model is still trained on `LogClosePrice`, but predictions are converted back using `np.exp()` before calculating MAPE and MdAPE.

The feature set used in this model is:

- LivingArea
- BathroomsTotalInteger
- BedroomsTotal
- Bed_Bath_Ratio
- Property_Age
- Months_From_Dec_2025
- Sqft_Per_Bedroom
- Lot_Utilization
- ZipMedianPrice

The target variable is:

- LogClosePrice

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Load Data and Prepare Features

In this section, the cleaned training and testing datasets are loaded.  
We create `LogClosePrice` as the modeling target and construct `ZipMedianPrice` using the median log closing price within each postal code from the training set.

Rows with missing values in the selected features or target are removed before model fitting.

In [2]:
tree_features = [
    'LivingArea',
    'BathroomsTotalInteger',
    'BedroomsTotal',
    'Bed_Bath_Ratio',
    'Property_Age',
    'Months_From_Dec_2025',
    'Sqft_Per_Bedroom',
    'Lot_Utilization',
    'ZipMedianPrice'
]

train = pd.read_csv('../processed_data/train_cleaned_fe.csv')
test = pd.read_csv('../processed_data/test_cleaned_fe.csv')

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice'] = np.log(test['ClosePrice'])

zip_median_price = train.groupby('PostalCode')['LogClosePrice'].median()

train['ZipMedianPrice'] = train['PostalCode'].map(zip_median_price)
test['ZipMedianPrice'] = test['PostalCode'].map(zip_median_price)

train = train.dropna(subset=tree_features + ['LogClosePrice'])
test = test.dropna(subset=tree_features + ['LogClosePrice'])

X_train = train[tree_features]
y_train = train['LogClosePrice']

X_test = test[tree_features]
y_test = test['LogClosePrice']

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## Randomized Hyperparameter Search

`RandomizedSearchCV` is used to test different hyperparameter combinations and select the best model based on cross-validated R².

In [3]:
xgb_tune_model = XGBRegressor(random_state=42)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3, 0.5]
}

random_search = RandomizedSearchCV(
    estimator=xgb_tune_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.3, 0.5],
                                        'learning_rate': [0.01, 0.03, 0.05, 0.1,
                                                          0.2],
                                        'max_depth': [3, 4, 5, 6, 8],
                                        'min_child_weight': [1, 3, 5],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, scoring='r2', verbose=1)

## Best Hyperparameters

After the randomized search finishes, the best hyperparameter combination is selected and used to train the final tuned XGBoost model.

In [4]:
print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

best_xgb_model = random_search.best_estimator_
best_xgb_model.fit(X_train, y_train)

Best Parameters: {'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV Score: 0.9039463438660847


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

## Evaluate the Tuned XGBoost Model

The tuned model is evaluated using:

- R² on the log scale
- R² on the actual price scale
- MAPE on the actual price scale
- MdAPE on the actual price scale

Following Marko's feedback, MAPE and MdAPE are computed after converting predictions and true values back to the original `ClosePrice` scale.

In [5]:
tuned_pred_train = best_xgb_model.predict(X_train)
tuned_pred_test = best_xgb_model.predict(X_test)

tuned_r2_train_log = r2_score(y_train, tuned_pred_train)
tuned_r2_test_log = r2_score(y_test, tuned_pred_test)

print(f"Train Set -- R^2 on Log Scale: {tuned_r2_train_log}")
print(f"Test Set -- R^2 on Log Scale: {tuned_r2_test_log}")

y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

tuned_pred_train_actual = np.exp(tuned_pred_train)
tuned_pred_test_actual = np.exp(tuned_pred_test)

tuned_r2_train_actual = r2_score(y_train_actual, tuned_pred_train_actual)
tuned_r2_test_actual = r2_score(y_test_actual, tuned_pred_test_actual)

print(f"Train Set -- R^2 on Actual Scale: {tuned_r2_train_actual}")
print(f"Test Set -- R^2 on Actual Scale: {tuned_r2_test_actual}")

tuned_mape_train = mean_absolute_percentage_error(y_train_actual, tuned_pred_train_actual)
tuned_mape_test = mean_absolute_percentage_error(y_test_actual, tuned_pred_test_actual)

print(f"Train MAPE: {tuned_mape_train * 100}%")
print(f"Test MAPE: {tuned_mape_test * 100}%")

tuned_mdape_train = np.median(np.abs((y_train_actual - tuned_pred_train_actual) / y_train_actual))
tuned_mdape_test = np.median(np.abs((y_test_actual - tuned_pred_test_actual) / y_test_actual))

print(f"Train MdAPE: {tuned_mdape_train * 100}%")
print(f"Test MdAPE: {tuned_mdape_test * 100}%")

Train Set -- R^2 on Log Scale: 0.9243288995756151
Test Set -- R^2 on Log Scale: 0.8949430048524374
Train Set -- R^2 on Actual Scale: 0.8928002281191354
Test Set -- R^2 on Actual Scale: 0.8385236990000339
Train MAPE: 12.5966650304812%
Test MAPE: 14.504524658334345%
Train MdAPE: 9.292761363636401%
Test MdAPE: 10.29366615853657%


In [6]:
# Initialize DataFrame
xg_boost_metrics = pd.DataFrame(
    index=[
        'Tuned XGBoost Model',
        'Log Range [12, 13)',
        'Log Range [13, 14)',
        'Log Range [14, 15)',
        'Log Range [15, 16)'
    ],
    columns=[
        'Train R^2', 'Test R^2',
        'Train R^2 (Log)', 'Test R^2 (Log)',
        'Train MAPE (%)', 'Test MAPE (%)',
        'Train MdAPE (%)', 'Test MdAPE (%)'
    ]
)

# Assign values (KEEP EVERYTHING NUMERIC)
xg_boost_metrics.loc['Tuned XGBoost Model'] = [
    tuned_r2_train_actual,
    tuned_r2_test_actual,
    tuned_r2_train_log,
    tuned_r2_test_log,
    tuned_mape_train * 100,
    tuned_mape_test * 100,
    tuned_mdape_train * 100,
    tuned_mdape_test * 100
]


# Optional: nicer display (does NOT change underlying data)
xg_boost_metrics = xg_boost_metrics.astype(float)

## Interpretation of Tuned Model Results

The tuned XGBoost model is evaluated on both the log scale and the actual price scale.

- R² on the log scale shows how well the model explains variation in `LogClosePrice`.
- R² on the actual scale shows how well the back-transformed predictions match real home prices.
- MAPE and MdAPE measure percentage error on the actual `ClosePrice` scale, which makes them more interpretable for housing prices.

A small gap between training and testing metrics suggests that the tuned model generalizes reasonably well.

# START: Vivian Lin

## Evaluate the Tuned Model on the [12, 13) Range

The hypertuned XGBoost model is evaluated on the subset of observations in the [12, 13) log-price intervals. The percentage-based metrics are computed based on `ClosePrice` scale after exponentiating predictions and true values

In [7]:
train_1213 = train[(train['LogClosePrice'] >= 12) & (train['LogClosePrice'] < 13)].copy()
test_1213 = test[(test['LogClosePrice'] >= 12) & (test['LogClosePrice'] < 13)].copy()

X_train_1213 = train_1213[tree_features]
y_train_1213 = train_1213['LogClosePrice']
y_train_1213_actual = train_1213['ClosePrice']

X_test_1213 = test_1213[tree_features]
y_test_1213 = test_1213['LogClosePrice']
y_test_1213_actual = test_1213['ClosePrice']


X_test_1213 = X_test_1213.reindex(columns=X_train_1213.columns, fill_value=0)

print("Train [12, 13) range shape:", X_train_1213.shape)
print("Test [12, 13) range shape:", X_test_1213.shape)

Train [12, 13) range shape: (6320, 9)
Test [12, 13) range shape: (1056, 9)


## XGModel Performance for the [12, 13) LogClosePrice Range

This section evaluates the tuned XGBoost model specifically on homes whose `LogClosePrice` falls within the interval **[12, 13)**.

This range-specific analysis helps us understand how well the model performs for this price band, rather than only reporting overall performance across all homes.

In [8]:
range_1213_pred_train = best_xgb_model.predict(X_train_1213)
range_1213_pred_test = best_xgb_model.predict(X_test_1213)

range_1213_r2_log_train = r2_score(y_train_1213, range_1213_pred_train)
range_1213_r2_log_test = r2_score(y_test_1213, range_1213_pred_test)

print(f'Range [12, 13) Train R^2 (log): {range_1213_r2_log_train}')
print(f'Range [12, 13) Test R^2 (log): {range_1213_r2_log_test}')

y_train_range1213_actual_pred = np.exp(range_1213_pred_train)
y_test_range1213_actual_pred = np.exp(range_1213_pred_test)

range1213_r2_train_actual = r2_score(y_train_1213_actual, y_train_range1213_actual_pred)
range1213_r2_test_actual = r2_score(y_test_1213_actual, y_test_range1213_actual_pred)

print(f'Range [12, 13) Train R^2 (Actual): {range1213_r2_train_actual}')
print(f'Range [12, 13) Test R^2 (Actual): {range1213_r2_test_actual}')

range_1213_mape_train = mean_absolute_percentage_error(y_train_1213_actual, y_train_range1213_actual_pred)
range_1213_mape_test = mean_absolute_percentage_error(y_test_1213_actual, y_test_range1213_actual_pred)

print(f"Range [12, 13) Train MAPE: {range_1213_mape_train * 100}%")
print(f"Range [12, 13) Test MAPE: {range_1213_mape_test * 100}%")

range_1213_mdape_train = np.median(np.abs((y_train_1213_actual - y_train_range1213_actual_pred) / y_train_1213_actual))
range_1213_mdape_test = np.median(np.abs((y_test_1213_actual - y_test_range1213_actual_pred) / y_test_1213_actual))

print(f"Range [13, 14) Train MdAPE: {range_1213_mdape_train * 100}%")
print(f"Range [13, 14) Test MdAPE: {range_1213_mdape_test * 100}%")

Range [12, 13) Train R^2 (log): -0.05550918516347636
Range [12, 13) Test R^2 (log): -0.3514696200137979
Range [12, 13) Train R^2 (Actual): -0.7138434024021061
Range [12, 13) Test R^2 (Actual): -1.244250584707105
Range [12, 13) Train MAPE: 15.861764896891192%
Range [12, 13) Test MAPE: 18.924962998868207%
Range [13, 14) Train MdAPE: 9.878317149653979%
Range [13, 14) Test MdAPE: 11.704116645152038%


In [9]:
xg_boost_metrics.loc['Log Range [12, 13)'] = [
    range1213_r2_train_actual,
    range1213_r2_test_actual,
    range_1213_r2_log_train,
    range_1213_r2_log_test,
    range_1213_mape_train * 100,
    range_1213_mape_test * 100,
    range_1213_mdape_train * 100,
    range_1213_mdape_test * 100
]

WHY ARE THE R^2 VALUES NEGATIVE??

## Interpretation for the [12, 13) Price Band

This section summarizes model performance for homes in the `[12, 13)` log-price range.

- If the R² values are negative. This could be due to the fact that this range is amongst the extreme low outliers. (I'm not too sure in what scenarios an $R^2$ value can be negative or if that's even possible)
- If MAPE and MdAPE are higher for datapoints within the range measured than the overall dataset. Predictions are further off from the actual price.
- Comparing these results with the full-dataset metrics helps determine whether the model performs better or worse for this specific segment of the market.

This range-based evaluation is useful for analyzing how the model performs on extremely low closing prices/outliers.

# END: Vivian Lin

## XGModel Performance for the [13, 14) LogClosePrice Range

This section evaluates the tuned XGBoost model specifically on homes whose `LogClosePrice` falls within the interval **[13, 14)**.

This range-specific analysis helps us understand how well the model performs for this price band, rather than only reporting overall performance across all homes.

In [10]:
train_1314 = train[(train['LogClosePrice'] >= 13) & (train['LogClosePrice'] < 14)].copy()
test_1314 = test[(test['LogClosePrice'] >= 13) & (test['LogClosePrice'] < 14)].copy()

X_train_1314 = train_1314[tree_features]
y_train_1314 = train_1314['LogClosePrice']
y_train_1314_actual = train_1314['ClosePrice']

X_test_1314 = test_1314[tree_features]
y_test_1314 = test_1314['LogClosePrice']
y_test_1314_actual = test_1314['ClosePrice']

X_test_1314 = X_test_1314.reindex(columns=X_train_1314.columns, fill_value=0)

print("Train [13, 14) shape:", X_train_1314.shape)
print("Test [13, 14) shape:", X_test_1314.shape)

Train [13, 14) shape: (39160, 9)
Test [13, 14) shape: (6122, 9)


## Evaluate the Tuned Model on the [13, 14) Range

The same tuned XGBoost model is applied to the subset of observations in the `[13, 14)` log-price interval.

Again, percentage-based metrics are computed on the actual `ClosePrice` scale after exponentiating predictions and true values.

In [11]:
pred_train_1314 = best_xgb_model.predict(X_train_1314)
pred_test_1314 = best_xgb_model.predict(X_test_1314)

r2_log_train_1314 = r2_score(y_train_1314, pred_train_1314)
r2_log_test_1314 = r2_score(y_test_1314, pred_test_1314)

pred_train_1314_actual = np.exp(pred_train_1314)
pred_test_1314_actual = np.exp(pred_test_1314)

r2_train_1314_actual = r2_score(y_train_1314_actual, pred_train_1314_actual)
r2_test_1314_actual = r2_score(y_test_1314_actual, pred_test_1314_actual)

mape_train_1314 = mean_absolute_percentage_error(y_train_1314_actual, pred_train_1314_actual)
mape_test_1314 = mean_absolute_percentage_error(y_test_1314_actual, pred_test_1314_actual)

mdape_train_1314 = np.median(np.abs((y_train_1314_actual - pred_train_1314_actual) / y_train_1314_actual))
mdape_test_1314 = np.median(np.abs((y_test_1314_actual - pred_test_1314_actual) / y_test_1314_actual))

print("XGBoost Results for LogClosePrice in [13, 14):")
print(f"Train R² (log): {r2_log_train_1314}")
print(f"Test R² (log): {r2_log_test_1314}")
print(f"Train R² (actual): {r2_train_1314_actual}")
print(f"Test R² (actual): {r2_test_1314_actual}")
print(f"Train MAPE (%): {mape_train_1314 * 100}")
print(f"Test MAPE (%): {mape_test_1314 * 100}")
print(f"Train MdAPE (%): {mdape_train_1314 * 100}")
print(f"Test MdAPE (%): {mdape_test_1314 * 100}")

XGBoost Results for LogClosePrice in [13, 14):
Train R² (log): 0.696301534069264
Test R² (log): 0.5976369112642219
Train R² (actual): 0.5897035401654672
Test R² (actual): 0.4336578995396445
Train MAPE (%): 11.10144640176512
Test MAPE (%): 12.539599322745607
Train MdAPE (%): 8.112212642731665
Test MdAPE (%): 8.834293305973018


In [12]:
xg_boost_metrics.loc['Log Range [13, 14)'] = [
    r2_train_1314_actual,
    r2_test_1314_actual,
    r2_log_train_1314,
    r2_log_test_1314,
    mape_train_1314 * 100,
    mape_test_1314 * 100,
    mdape_train_1314 * 100,
    mdape_test_1314 * 100
]

## Interpretation for the [13, 14) Price Band

This section summarizes model performance for homes in the `[13, 14)` log-price range.

- If the R² values remain relatively high, the model explains a substantial portion of variation even within this narrower price band.
- If MAPE and MdAPE are low, predictions are relatively close to actual prices in percentage terms.
- Comparing these results with the full-dataset metrics helps determine whether the model performs better or worse for this specific segment of the market.

This range-based evaluation is useful for identifying whether model performance differs across different home price levels.

### END: Liz Choi

## START Steven Shi

## XGModel Performance for the [14, 15) LogClosePrice Range

This section evaluates the tuned XGBoost model specifically on homes whose LogClosePrice falls within the interval [14, 15).


In [13]:
# Filtering and preparing data for log price range [14, 15)

train_1415 = train[(train['LogClosePrice'] >= 14) & (train['LogClosePrice'] < 15)].copy()
test_1415 = test[(test['LogClosePrice'] >= 14) & (test['LogClosePrice'] < 15)].copy()

X_train_1415 = train_1415[tree_features]
y_train_1415 = train_1415['LogClosePrice']
y_train_1415_actual = train_1415['ClosePrice']

X_test_1415 = test_1415[tree_features]
y_test_1415 = test_1415['LogClosePrice']
y_test_1415_actual = test_1415['ClosePrice']

X_test_1415 = X_test_1415.reindex(columns=X_train_1415.columns, fill_value=0)


print("Train range shape:", X_train_1415.shape)
print("Test range shape:", X_test_1415.shape)

Train range shape: (19478, 9)
Test range shape: (2695, 9)


In [14]:
# Predict
pred_train_1415 = best_xgb_model.predict(X_train_1415)
pred_test_1415 = best_xgb_model.predict(X_test_1415)

# R² log
r2_log_train_1415 = r2_score(y_train_1415, pred_train_1415)
r2_log_test_1415 = r2_score(y_test_1415, pred_test_1415)

# Convert to actual
pred_train_1415_actual = np.exp(pred_train_1415)
pred_test_1415_actual = np.exp(pred_test_1415)

# R² actual
r2_train_1415_actual = r2_score(y_train_1415_actual, pred_train_1415_actual)
r2_test_1415_actual = r2_score(y_test_1415_actual, pred_test_1415_actual)

# MAPE
mape_train_1415 = mean_absolute_percentage_error(y_train_1415_actual, pred_train_1415_actual)
mape_test_1415 = mean_absolute_percentage_error(y_test_1415_actual, pred_test_1415_actual)

# MdAPE
mdape_train_1415 = np.median(np.abs((y_train_1415_actual - pred_train_1415_actual) / y_train_1415_actual))
mdape_test_1415 = np.median(np.abs((y_test_1415_actual - pred_test_1415_actual) / y_test_1415_actual))

# Print metrics
print("XGBoost Results for LogClosePrice in [14, 15):")
print(f"Train R² (log): {r2_log_train_1415}")
print(f"Test R² (log): {r2_log_test_1415}")
print(f"Train R² (actual): {r2_train_1415_actual}")
print(f"Test R² (actual): {r2_test_1415_actual}")
print(f"Train MAPE (%): {mape_train_1415 * 100}")
print(f"Test MAPE (%): {mape_test_1415 * 100}")
print(f"Train MdAPE (%): {mdape_train_1415 * 100}")
print(f"Test MdAPE (%): {mdape_test_1415 * 100}")


XGBoost Results for LogClosePrice in [14, 15):
Train R² (log): 0.48630748577429983
Test R² (log): 0.26610311817644994
Train R² (actual): 0.5353616362877547
Test R² (actual): 0.3169282597289731
Train MAPE (%): 13.889608190773487
Test MAPE (%): 16.19152795667352
Train MdAPE (%): 11.333852280556703
Test MdAPE (%): 12.644480453716833


In [15]:
xg_boost_metrics.loc['Log Range [14, 15)'] = [
    r2_train_1415_actual,
    r2_test_1415_actual,
    r2_log_train_1415,
    r2_log_test_1415,
    mape_train_1415 * 100,
    mape_test_1415 * 100,
    mdape_train_1415 * 100,
    mdape_test_1415 * 100
]
xg_boost_metrics

,Train R^2,Test R^2,Train R^2 (Log),Test R^2 (Log),Train MAPE (%),Test MAPE (%),Train MdAPE (%),Test MdAPE (%)
Tuned XGBoost Model,0.892800,0.838524,0.924329,0.894943,12.596665,14.504525,9.292761,10.293666
"Log Range [12, 13)",-0.713843,-1.244251,-0.055509,-0.351470,15.861765,18.924963,9.878317,11.704117
"Log Range [13, 14)",0.589704,0.433658,0.696302,0.597637,11.101446,12.539599,8.112213,8.834293
"Log Range [14, 15)",0.535362,0.316928,0.486307,0.266103,13.889608,16.191528,11.333852,12.644480
"Log Range [15, 16)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Interpretation for the [14, 15) Price Band

#### Overall Model Performance
The tuned XGBoost model performs strongly on the full dataset:
* High R² (≈ 0.89 test) → explains most of the variance;
* MAPE ~14.5% → reasonable prediction error;
* Small gap between train and test → good generalization, limited overfitting.

Overall, the model is reliable and well-tuned.

#### Performance on Price Range [14, 15)
Performance drops significantly:
* R² (test) falls to ~0.60 (log) / ~0.43 (actual) → much weaker fit;
* Errors (MAPE ~12.5%) are not worse—and even slightly lower than overall;
* Train vs test gap is larger → more instability in this segment.

## XGModel Performance for the [15, 16) LogClosePrice Range
This section evaluates the tuned XGBoost model specifically on homes whose LogClosePrice falls within the interval [15, 16).

In [16]:
train_1516 = train[(train['LogClosePrice'] >= 15) & (train['LogClosePrice'] < 16)].copy()
test_1516 = test[(test['LogClosePrice'] >= 15) & (test['LogClosePrice'] < 16)].copy()

X_train_1516 = train_1516[tree_features]
y_train_1516 = train_1516['LogClosePrice']
y_train_1516_actual = train_1516['ClosePrice']

X_test_1516 = test_1516[tree_features]
y_test_1516 = test_1516['LogClosePrice']
y_test_1516_actual = test_1516['ClosePrice']

X_test_1516 = X_test_1516.reindex(columns=X_train_1516.columns, fill_value=0)

In [17]:
pred_train_1516 = best_xgb_model.predict(X_train_1516)
pred_test_1516 = best_xgb_model.predict(X_test_1516)

# R² log
r2_log_train_1516 = r2_score(y_train_1516, pred_train_1516)
r2_log_test_1516 = r2_score(y_test_1516, pred_test_1516)

# Convert to actual
pred_train_1516_actual = np.exp(pred_train_1516)
pred_test_1516_actual = np.exp(pred_test_1516)

# R² actual
r2_train_1516_actual = r2_score(y_train_1516_actual, pred_train_1516_actual)
r2_test_1516_actual = r2_score(y_test_1516_actual, pred_test_1516_actual)

# MAPE
mape_train_1516 = mean_absolute_percentage_error(y_train_1516_actual, pred_train_1516_actual)
mape_test_1516 = mean_absolute_percentage_error(y_test_1516_actual, pred_test_1516_actual)

# MdAPE
mdape_train_1516 = np.median(np.abs((y_train_1516_actual - pred_train_1516_actual) / y_train_1516_actual))
mdape_test_1516 = np.median(np.abs((y_test_1516_actual - pred_test_1516_actual) / y_test_1516_actual))

# Print metrics
print("XGBoost Results for LogClosePrice in [15, 16):")
print(f"Train R² (log): {r2_log_train_1516}")
print(f"Test R² (log): {r2_log_test_1516}")
print(f"Train R² (actual): {r2_train_1516_actual}")
print(f"Test R² (actual): {r2_test_1516_actual}")
print(f"Train MAPE (%): {mape_train_1516 * 100}")
print(f"Test MAPE (%): {mape_test_1516 * 100}")
print(f"Train MdAPE (%): {mdape_train_1516 * 100}")
print(f"Test MdAPE (%): {mdape_test_1516 * 100}")

XGBoost Results for LogClosePrice in [15, 16):
Train R² (log): -0.3850525303498471
Test R² (log): -1.1500659307224486
Train R² (actual): 0.18771071789464044
Test R² (actual): -0.2156485168335338
Train MAPE (%): 17.293597331712245
Test MAPE (%): 21.99073776430889
Train MdAPE (%): 14.492475917729758
Test MdAPE (%): 19.98573848382083


In [18]:
xg_boost_metrics.loc['Log Range [15, 16)'] = [
    r2_train_1516_actual,
    r2_test_1516_actual,
    r2_log_train_1516,
    r2_log_test_1516,
    mape_train_1516 * 100,
    mape_test_1516 * 100,
    mdape_train_1516 * 100,
    mdape_test_1516 * 100
]
xg_boost_metrics

,Train R^2,Test R^2,Train R^2 (Log),Test R^2 (Log),Train MAPE (%),Test MAPE (%),Train MdAPE (%),Test MdAPE (%)
Tuned XGBoost Model,0.892800,0.838524,0.924329,0.894943,12.596665,14.504525,9.292761,10.293666
"Log Range [12, 13)",-0.713843,-1.244251,-0.055509,-0.351470,15.861765,18.924963,9.878317,11.704117
"Log Range [13, 14)",0.589704,0.433658,0.696302,0.597637,11.101446,12.539599,8.112213,8.834293
"Log Range [14, 15)",0.535362,0.316928,0.486307,0.266103,13.889608,16.191528,11.333852,12.644480
"Log Range [15, 16)",0.187711,-0.215649,-0.385053,-1.150066,17.293597,21.990738,14.492476,19.985738


## Interpretation for the [15, 16) Price Band

### Performance on High-Price Range [15, 16)
Performance breaks down significantly in this segment:
* R² becomes negative (test: -1.18 log, -0.22 actual)
→ worse than simply predicting the mean;
* MAPE jumps to ~22% → much larger prediction errors
* MdAPE ~19.6% → typical error is also high
* Large train–test gap → poor generalization

## END Steven Shi

In [19]:
xg_boost_metrics

,Train R^2,Test R^2,Train R^2 (Log),Test R^2 (Log),Train MAPE (%),Test MAPE (%),Train MdAPE (%),Test MdAPE (%)
Tuned XGBoost Model,0.892800,0.838524,0.924329,0.894943,12.596665,14.504525,9.292761,10.293666
"Log Range [12, 13)",-0.713843,-1.244251,-0.055509,-0.351470,15.861765,18.924963,9.878317,11.704117
"Log Range [13, 14)",0.589704,0.433658,0.696302,0.597637,11.101446,12.539599,8.112213,8.834293
"Log Range [14, 15)",0.535362,0.316928,0.486307,0.266103,13.889608,16.191528,11.333852,12.644480
"Log Range [15, 16)",0.187711,-0.215649,-0.385053,-1.150066,17.293597,21.990738,14.492476,19.985738
